# 키워드 기반 희소 검색(Sparse Search)

## 환경 설정

In [1]:
# 필요한 클래스 임포트
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.retrievers import BM25Retriever
from mecab import MeCab

C:\Users\playdata2\AppData\Local\Temp\ipykernel_18064\84092111.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


## 데이터 불러오기

In [2]:
# 데이터 로드 및 변환 도구 생성 -> TextLoader 클래스 호출
loader = TextLoader(
    file_path='manual.txt',
    encoding='utf-8'
)

# load() 함수 호출, Document 자료형 생성
documents = loader.load()
print(f'documents: {documents}')

documents: [Document(metadata={'source': 'manual.txt'}, page_content="질문: 코드크래프터스 아카데미에 어떻게 가입하나요?\n답변: 저희 웹사이트 우측 상단의 '회원가입' 버튼을 클릭하시고, 이메일 주소와 비밀번호만 입력하면 간단하게 가입이 완료됩니다. 이메일 인증 후 바로 모든 서비스를 이용하실 수 있습니다.\n\n질문: 수강 신청한 강의를 환불하고 싶어요. 환불 정책은 어떻게 되나요?\n답변: 네, 환불 가능합니다. 결제일로부터 7일 이내, 그리고 강의 수강률이 5% 미만일 경우 전액 환불해 드리고 있습니다. 고객센터를 통해 문의하시면 신속하게 처리해 드리겠습니다.\n\n질문: 강의를 다 들으면 수료증을 받을 수 있나요?\n답변: 물론입니다. 각 강의의 모든 챕터를 100% 수강 완료하시면, 즉시 온라인으로 발급되는 공식 수료증을 받으실 수 있습니다. 수료증은 PDF 파일로 다운로드하여 소장하거나 제출용으로 활용하실 수 있습니다.\n\n질문: 어떤 결제 수단을 사용할 수 있나요?\n답변: 저희는 학습자님의 편의를 위해 신용카드, 실시간 계좌이체, 그리고 가상계좌 입금 등 다양한 결제 방법을 지원하고 있습니다. 원하시는 결제 수단을 선택하여 진행해 주세요.\n\n질문: 강의 동영상이 자꾸 끊기거나 재생되지 않아요.\n답변: 동영상 재생에 불편을 드려 죄송합니다. 우선 사용하고 계신 인터넷 브라우저의 캐시를 삭제해 보시거나, 다른 브라우저(크롬, 엣지 등)에서 재생해 보시길 권장합니다. 문제가 지속될 경우, 즉시 기술 지원팀에 문의해 주세요.\n\n질문: 강의 자료나 소스 코드는 어디서 다운로드받나요?\n답변: 각 강의의 첫 번째 챕터인 '강의 소개' 섹션과 각 챕터의 하단에서 관련 PDF 교안 및 전체 소스 코드를 압축 파일 형태로 다운로드하실 수 있습니다.\n\n질문: 수강 기간에 제한이 있나요?\n답변: 아니요, 저희 코드크래프터스 아카데미의 모든 강의는 한 번 구매하시면 평생 소장이 가능합니다. 시간

## 텍스트 분할

In [3]:
# 데이터 분할 도구 생성
text_splitter = RecursiveCharacterTextSplitter(    
    chunk_size=160,
    chunk_overlap=0
)

# split_documents() 함수 함수, 분할된 문서 생성
docs = text_splitter.split_documents(documents)
print(f'분할된 원본 문서: \n{docs}')
print('-'*80)
print(f'분할된 문서 조각의 수: {len(docs)}')
print('-'*80)
print(f'분할된 문서 조각 예시: \n{docs[0]}')

분할된 원본 문서: 
[Document(metadata={'source': 'manual.txt'}, page_content="질문: 코드크래프터스 아카데미에 어떻게 가입하나요?\n답변: 저희 웹사이트 우측 상단의 '회원가입' 버튼을 클릭하시고, 이메일 주소와 비밀번호만 입력하면 간단하게 가입이 완료됩니다. 이메일 인증 후 바로 모든 서비스를 이용하실 수 있습니다."), Document(metadata={'source': 'manual.txt'}, page_content='질문: 수강 신청한 강의를 환불하고 싶어요. 환불 정책은 어떻게 되나요?\n답변: 네, 환불 가능합니다. 결제일로부터 7일 이내, 그리고 강의 수강률이 5% 미만일 경우 전액 환불해 드리고 있습니다. 고객센터를 통해 문의하시면 신속하게 처리해 드리겠습니다.'), Document(metadata={'source': 'manual.txt'}, page_content='질문: 강의를 다 들으면 수료증을 받을 수 있나요?\n답변: 물론입니다. 각 강의의 모든 챕터를 100% 수강 완료하시면, 즉시 온라인으로 발급되는 공식 수료증을 받으실 수 있습니다. 수료증은 PDF 파일로 다운로드하여 소장하거나 제출용으로 활용하실 수 있습니다.'), Document(metadata={'source': 'manual.txt'}, page_content='질문: 어떤 결제 수단을 사용할 수 있나요?\n답변: 저희는 학습자님의 편의를 위해 신용카드, 실시간 계좌이체, 그리고 가상계좌 입금 등 다양한 결제 방법을 지원하고 있습니다. 원하시는 결제 수단을 선택하여 진행해 주세요.'), Document(metadata={'source': 'manual.txt'}, page_content='질문: 강의 동영상이 자꾸 끊기거나 재생되지 않아요.\n답변: 동영상 재생에 불편을 드려 죄송합니다. 우선 사용하고 계신 인터넷 브라우저의 캐시를 삭제해 보시거나, 다른 브라우저(크롬, 엣지 등)에서 재생해 보시길 권장합니다. 문제가 지속

## MeCab 형태소 분석기 기반 토큰화 함수 정의

In [4]:
# MeCab 클래스 호출 -> 형태소 분석 도구 생성
mecab = MeCab()

In [ ]:
# 토큰화 함수 정의
def mecab_tokenizer(text):
    """
    Mecab 형태소 분석기를 사용하여 텍스트에서 의미 있는 형태소(명사, 용언, 숫자)만 추출합니다.
    - N: 명사 (NNG, NNP 등)
    - V: 용언 (VV: 동사, VA: 형용사 등)
    - S: 숫자 및 외국어, 기호 등 (SN, SL 등)
    """

    ## 의미 있는 형태소 추출 결과 저장할 빈 리스트 생성
    tokens = []

    ## 품사 태깅 -> 결과: [('한글', 'NNG'), ...]
    tags = mecab.pos(text)

    ## for문 정의
    for




